In [1]:
import json
import os
import numpy as np
import cupy as cp

from model import GoePT
from dataset import Dataset


with open(os.path.join("../..", "checkpoints", 'class_rock_pop_80percent.json'), mode='r', encoding='utf-8') as in_file:
    state_dict = json.load(in_file)

model = GoePT.from_state_dict(state_dict)

In [2]:
only_genres = ["pop", "rock", "classical"]
test_dataset = Dataset("test", only_genres=only_genres, uniform=True, data_dir="../../data", context_length=model.context_length)
test_dataset.get_slices(context_length=model.context_length, data_dir='../../data') # precompute slices, they stay cached

print("Dataset loaded.")

Dataset loaded.


In [3]:
rng = np.random.default_rng()

top1_correct = 0
samples_seen = 0

for _ in range(20):
    x, y = test_dataset.get_batch_from_slices(model.batch_size, rng=rng)
    x, y = cp.asarray(x), cp.asarray(y)
    logits, loss = model.forward(x,targets=y,train= False)

    top1_correct += (cp.argmax(logits,-1).flatten()==y).sum().item()
    samples_seen += y.size

print(f"Top1 accuracy: {top1_correct/samples_seen:.4f} ({top1_correct}/{samples_seen})")

Top1 accuracy: 0.8383 (2012/2400)
